# Chapter 4: Spark SQL Engine and Performance Tuning

## Setup

Create a local Spark session and small fact and dimension DataFrames for the examples. `local[*]` uses the available processor cores on this machine; it is for learning, not a cluster deployment.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Spark SQL Engine and Performance Tuning")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

orders = spark.createDataFrame(
    [
        (1, "Ava", "North", "books", 120.0),
        (2, "Ben", "South", "games", 75.0),
        (3, "Ava", "North", "games", 210.0),
        (4, "Chen", "West", "books", 90.0),
        (5, "Diya", "North", "books", 60.0),
    ],
    ["order_id", "customer", "region", "category", "amount"],
)

regions = spark.createDataFrame(
    [("North", "N"), ("South", "S"), ("West", "W")],
    ["region", "region_code"],
)

orders.show()

## Spark SQL Engine

The **Spark SQL engine** executes structured-data operations expressed through SQL or the DataFrame API. These two interfaces use the same underlying query-planning and execution machinery, so equivalent operations can be optimized similarly.

At a high level, Spark parses the request, resolves tables and columns, builds a **logical plan** describing what result is needed, optimizes that plan, and selects a **physical plan** describing how to run it. The physical plan runs as distributed tasks on executors. Catalyst is Spark SQL's optimizer; execution may use JVM code generation and other engine optimizations. The exact operators and plan depend on the query and runtime settings.

Prefer built-in DataFrame and SQL expressions where possible: Spark can inspect and optimize them more effectively than opaque Python code such as a Python UDF.

In [ ]:
orders.createOrReplaceTempView("orders")

sql_result = spark.sql("""
    SELECT region, SUM(amount) AS total_amount
    FROM orders
    WHERE amount > 75
    GROUP BY region
""")

sql_result.explain("formatted")
sql_result.show()

## Driver Memory Management

The **driver** runs the application’s main logic, plans jobs, and coordinates executors. Driver memory is used for query plans, task and executor metadata, job results returned to the driver, and other application-side objects. It is not the place to store the entire distributed dataset.

A common driver-memory problem is bringing too much data to the driver with `collect()` or converting a large result to Pandas. Prefer distributed processing and writes, or return a deliberately bounded sample. Configure driver memory at application submission/startup (for example, with `spark-submit --driver-memory`); changing a Spark property after the driver JVM has started cannot enlarge that JVM's heap.

In [ ]:
# Safe for inspection: only a few rows are returned for display.
orders.select("order_id", "customer", "amount").show(3)

# Avoid this pattern on a large DataFrame: all rows are copied to driver memory.
# all_rows = orders.collect()

# Example startup setting (set before launching the application):
# spark-submit --driver-memory 4g my_application.py

## Executor Memory Management

An **executor** is an application-specific process on a worker node. Executor memory is used by Spark tasks, intermediate data such as shuffle and aggregation state, cached or persisted data, and (depending on the workload) Python worker processes and other native/off-heap allocations.

A task can run out of memory even when the whole dataset seems small if one partition is very large or skewed. Increasing executor heap alone may not solve the problem: also check partition sizes, concurrent tasks per executor, skew, caching, and memory outside the JVM heap. Executor memory and memory overhead are generally selected before the application starts and depend on the cluster manager.

In [ ]:
# Inspect the current Spark context settings that mention memory.
memory_settings = [
    (key, value)
    for key, value in spark.sparkContext.getConf().getAll()
    if "memory" in key.lower()
]
print("Memory-related settings explicitly present in this SparkConf:")
for key, value in memory_settings:
    print(f"{key} = {value}")

# Example startup settings; use values appropriate for the cluster:
# spark-submit --executor-memory 4g --conf spark.executor.memoryOverhead=1g my_application.py

## Unified Memory Management

Spark's **unified memory region** is a managed portion of executor JVM heap shared by two purposes: **execution memory** (for operations such as joins, sorts, and aggregations) and **storage memory** (for cached or persisted blocks). Sharing lets one purpose use otherwise available space rather than permanently reserving all of it for the other.

| Execution memory | Storage memory |
| --- | --- |
| Supports computations such as joins, sorts, and aggregations. | Holds cached or persisted data blocks. |
| Can use available space in the shared region; execution may reclaim space occupied by storage when needed. | Can use available space, but cached blocks may be evicted under memory pressure. |
| Large shuffles or aggregations can raise its demand. | Caching more data than available memory can lead to eviction or disk use, depending on the storage level. |

This describes the JVM's managed memory, not all process memory. Python workers, JVM overhead, native memory, and other allocations also consume container resources. Exact behavior and defaults are version/configuration dependent.

In [ ]:
# A grouping can require execution memory; persistence uses storage memory.
grouped = orders.groupBy("region").agg(F.sum("amount").alias("total_amount"))
grouped.show()

cached_orders = orders.persist()
cached_orders.count()  # Materializes persisted data when the action runs.
cached_orders.unpersist()

## Executor Out of Memory

**Out of memory (OOM)** means a process or container could not obtain enough memory for its work. A JVM heap OOM, an operating-system/container kill due to memory overhead, and Python-worker memory exhaustion can look different in logs, so diagnose the actual failure before changing settings.

| Common cause | First things to investigate |
| --- | --- |
| A few oversized or skewed partitions | Compare task input sizes and durations; increase or rebalance partitions, or address skew. |
| Large data collected to the driver | Avoid `collect()`/large `toPandas()`; aggregate or write results in a distributed way. |
| Large joins, sorts, or aggregations | Inspect shuffle size, reduce input columns/rows, and review join strategy and partitioning. |
| Excessive cached data | Persist only reused data; choose a suitable storage level and `unpersist()` when finished. |
| Non-heap or Python memory exceeds container allowance | Check executor logs, container diagnostics, Python worker usage, and configured memory overhead. |

Increasing executor memory may be appropriate after diagnosis, but it consumes more cluster resources and does not fix skew or driver-side collection. More partitions can reduce per-task working data, but too many partitions add scheduling overhead.

In [ ]:
# Inspect how much parallel work Spark has for this DataFrame.
print("Input partitions:", orders.rdd.getNumPartitions())

# For a substantially larger input, repartitioning may lower per-task data volume.
# Validate partition sizes and shuffle costs before choosing a production value.
balanced_orders = orders.repartition(4, "region")
print("Partitions after repartition:", balanced_orders.rdd.getNumPartitions())

## Salting in PySpark

**Salting** is a technique for reducing join or aggregation skew when one or a few keys occur much more often than others. The hot key is split into several artificial subkeys (the original key plus a salt value), allowing its records to spread across more partitions.

For a join, both sides must use compatible salted keys. A common pattern is to assign each fact row a salt and replicate only the matching small dimension row across all salt values. The join then uses both the original key and salt. Salting increases data and join work, so apply it only to identified hot keys, choose a modest salt count, and verify that results are not duplicated or lost.

In [ ]:
salt_buckets = 4

# Assign a deterministic salt to each fact row. In real data, choose a stable
# row identifier and salt only the skewed key(s) where possible.
salted_orders = orders.withColumn(
    "salt",
    F.pmod(F.hash("order_id"), F.lit(salt_buckets)),
)

# Replicate each matching dimension row over all salt values so every salted
# fact row can still find its one corresponding dimension row.
salt_values = spark.range(salt_buckets).select(F.col("id").alias("salt"))
salted_regions = regions.crossJoin(salt_values)

salted_join = salted_orders.join(
    salted_regions,
    on=["region", "salt"],
    how="inner",
)
salted_join.select("order_id", "region", "region_code", "salt").show()

## Cache and Persist in Apache Spark

**Caching** keeps a DataFrame or RDD's computed partitions available for reuse. In Spark, caching is a convenient form of persistence: both mark data to be retained after computation, and neither computes it immediately. A later action materializes the stored partitions, which may be evicted if resources are needed.

| `cache()` | `persist()` |
| --- | --- |
| Convenient method that uses the API's default storage level. | Method that lets you specify a storage level, such as memory-only or memory-and-disk, where supported. |
| Best when the default storage behavior is suitable. | Best when you need to control how cached partitions are stored. |
| For DataFrames, equivalent to calling `persist()` with the default DataFrame storage level. | For DataFrames, `persist()` with no argument also uses that default; with an explicit level, storage behavior can differ. |

Persist data when it is expensive to compute and will be reused. Avoid storing one-use or excessively large datasets. Call `unpersist()` when the data is no longer needed so executor resources can be reclaimed. Exact default levels and available storage-level names depend on the Spark API and version.


In [ ]:
reusable = orders.filter(F.col("amount") >= 75).persist()

# The action computes the rows and materializes their persisted partitions.
print("Reusable row count:", reusable.count())

# Reuse the cached result for another operation.
reusable.groupBy("region").count().show()

# Release persisted blocks when finished.
reusable.unpersist()

## Edge Node and Deployment Mode

An **edge node** (often called a gateway node) is a machine near the cluster where users can submit jobs and access cluster services. It is not itself a Spark cluster-manager role. Running a driver on an edge node in client mode can keep it near executors and provide interactive console output.

**Deploy mode** specifies where the driver process runs:

| Client deploy mode | Cluster deploy mode |
| --- | --- |
| The driver runs in the submitting process, often on an edge/gateway node. | The cluster manager launches the driver within the cluster. |
| The submitter remains connected to the driver; useful for interactive work when the submit host is near the cluster. | Useful when submitting remotely so the driver runs near executors. |
| The submit host must remain available for the driver for the application's lifetime. | Driver lifecycle is managed in the cluster according to the cluster manager. |

Support and details vary by cluster manager and application type. Choose a mode supported by your deployment and keep the driver network-accessible to executors.

In [ ]:
print("Current master:", spark.sparkContext.master)
print("Current application:", spark.sparkContext.appName)

# Example submission commands (replace yarn and the script path as needed):
print("Client mode:  spark-submit --master yarn --deploy-mode client app.py")
print("Cluster mode: spark-submit --master yarn --deploy-mode cluster app.py")

## Dynamic Partition Pruning in Apache Spark

**Partition pruning** skips reading data files from partitions that cannot contribute to a query result. Static pruning can use a partition filter known before execution. **Dynamic partition pruning (DPP)** uses values discovered from another side of a join at runtime to avoid scanning irrelevant partitions of a partitioned data source.

DPP is especially useful for a large fact table physically partitioned by a join key, joined to a smaller dimension table filtered by a predicate. It is not guaranteed to apply: Spark needs a suitable query shape and source, and the optimizer may decide pruning is not beneficial. Check the physical plan and scan metrics to confirm.

In [ ]:
import tempfile
from pathlib import Path

spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")

# A partitioned file source is needed to demonstrate partition pruning.
with tempfile.TemporaryDirectory() as temp_dir:
    fact_path = str(Path(temp_dir) / "orders_by_region")
    orders.write.mode("overwrite").partitionBy("region").parquet(fact_path)
    partitioned_orders = spark.read.parquet(fact_path)

    selected_regions = regions.filter(F.col("region") == "North")
    pruned_query = partitioned_orders.join(selected_regions, on="region")
    pruned_query.explain("formatted")
    pruned_query.show()

## Adaptive Query Execution (AQE)

**Adaptive Query Execution (AQE)** can revise parts of a physical query plan while a query is running, using runtime statistics collected at execution boundaries such as shuffles. It complements the pre-execution optimizer by adapting decisions when actual data sizes differ from estimates.

AQE can, when applicable and enabled, coalesce small shuffle partitions, change some join strategies (for example, convert a sort-merge join to a broadcast join when runtime statistics allow), and optimize skewed joins. These are conditional optimizations, not guarantees for every query. Inspect the final plan and Spark UI to see which adaptations occurred. AQE availability and defaults depend on the Spark version and configuration.

In [ ]:
# Enable AQE for this Spark session; in many modern Spark versions it is enabled by default.
spark.conf.set("spark.sql.adaptive.enabled", "true")

adaptive_query = (
    orders.repartition(8, "region")
    .groupBy("region")
    .agg(F.sum("amount").alias("total_amount"))
)

adaptive_query.show()  # Execute first so runtime statistics are available.
adaptive_query.explain("formatted")  # Inspect the plan; details vary by version.